<div class="cover">

# Lab 5 — Text Representation

**Name:** Hadi Muneer Abu Allairat<br>
**Student ID:** 2230005761<br>
**Datasets:** toy sentence sets (Tasks 1–2), `simpsons_script_lines.csv` (Tasks 3–5)

</div>

The lab covers two ways of turning text into vectors. TF-IDF builds one axis per vocabulary term and
weights it by how rare the term is across the corpus; Word2Vec learns a dense vector per word from
the company it keeps. The first two tasks use TF-IDF and cosine similarity on small hand-written
sentence sets, where every number can be checked by hand. Tasks 3–5 move to a real corpus — 127k
lines of Simpsons dialogue — where the interesting question is whether the learned vectors actually
encode anything about the show.

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## Task 1 — Cosine similarity

Four sentences, deliberately built so that some share most of their words and others barely overlap.
Cosine similarity measures the angle between the TF-IDF vectors, so it compares *direction* rather
than *length* — a long document and a short one about the same topic still score high.

In [2]:
docs = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(docs)

print("vocabulary:", list(vectorizer.get_feature_names_out()))
print("matrix shape:", vectors.shape)

vocabulary: ['and', 'document', 'first', 'is', 'one', 'second', 'the', 'third', 'this']
matrix shape: (4, 9)


In [3]:
similarity = cosine_similarity(vectors)

labels = [f"doc {i + 1}" for i in range(len(docs))]
pd.DataFrame(similarity, index=labels, columns=labels).round(3)

,doc 1,doc 2,doc 3,doc 4
doc 1,1.000,0.647,0.308,1.000
doc 2,0.647,1.000,0.225,0.647
doc 3,0.308,0.225,1.000,0.308
doc 4,1.000,0.647,0.308,1.000


The diagonal is 1.0 by construction — every vector is perfectly similar to itself.

The pair worth looking at is documents 1 and 4, which score **exactly 1.000** — the same as the
diagonal. The model considers them not merely similar but identical, even though one is a statement
and the other a question: *"This is the first document."* and *"Is this the first document?"* contain
exactly the same words in a different order, and the question mark is dropped by the default
tokenizer. TF-IDF is a bag-of-words model, so word order is discarded entirely and the two sentences
produce byte-identical vectors. That is a genuine limitation rather than a quirk of this example —
"dog bites man" and "man bites dog" would also score 1.000.

Document 3 is the odd one out against all three others. It shares only the stopwords *and*, *is*,
*this* with them, and those are exactly the terms IDF pushes toward zero, so little weight survives.

In [4]:
pairs = [
    (i, j)
    for i in range(len(docs))
    for j in range(i + 1, len(docs))
]
ranked = sorted(pairs, key=lambda p: similarity[p], reverse=True)

for i, j in ranked:
    print(f"{similarity[i, j]:.3f}   doc {i + 1} vs doc {j + 1}")
    print(f"          {docs[i]}")
    print(f"          {docs[j]}")

1.000   doc 1 vs doc 4
          This is the first document.
          Is this the first document?
0.647   doc 1 vs doc 2
          This is the first document.
          This document is the second document.
0.647   doc 2 vs doc 4
          This document is the second document.
          Is this the first document?
0.308   doc 1 vs doc 3
          This is the first document.
          And this is the third one.
0.308   doc 3 vs doc 4
          And this is the third one.
          Is this the first document?
0.225   doc 2 vs doc 3
          This document is the second document.
          And this is the third one.


## Task 2 — TF-IDF and important words

Three sentences about data science. The question is which words TF-IDF considers important in each,
and the answer is governed by the IDF term: a word that appears in *every* document carries no
discriminating information, so its weight is driven down regardless of how often it is repeated.

In [5]:
sentences = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data",
]

tfidf = TfidfVectorizer()
matrix = tfidf.fit_transform(sentences)
terms = tfidf.get_feature_names_out()

frame = pd.DataFrame(matrix.toarray(), columns=terms, index=[f"s{i + 1}" for i in range(3)])
frame.round(3)

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
s1,0.000,0.0,0.0,0.190,0.321,0.321,0.244,0.321,0.488,0.244,0.488,0.000,0.244,0.0
s2,0.000,0.4,0.4,0.236,0.000,0.000,0.304,0.000,0.304,0.304,0.304,0.000,0.304,0.4
s3,0.543,0.0,0.0,0.641,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.543,0.000,0.0


In [6]:
for i, sentence in enumerate(sentences):
    row = frame.iloc[i]
    top = row[row > 0].sort_values(ascending=False).head(4)
    print(f"s{i + 1}: {sentence}")
    for term, weight in top.items():
        print(f"      {weight:.3f}  {term}")
    print()

s1: data science is one of the most important fields of science
      0.488  science
      0.488  of
      0.321  most
      0.321  fields

s2: this is one of the best data science courses
      0.400  best
      0.400  courses
      0.400  this
      0.304  of

s3: data scientists analyze data
      0.641  data
      0.543  analyze
      0.543  scientists



In [7]:
# Document frequency and IDF for every term, to show where the weights come from.
counts = (matrix > 0).sum(axis=0).A1
pd.DataFrame({
    "term": terms,
    "appears_in_docs": counts,
    "idf": tfidf.idf_.round(3),
}).sort_values(["appears_in_docs", "idf"]).reset_index(drop=True)

,term,appears_in_docs,idf
0,analyze,1,1.693
1,best,1,1.693
2,courses,1,1.693
3,fields,1,1.693
4,important,1,1.693
5,most,1,1.693
6,scientists,1,1.693
7,this,1,1.693
8,is,2,1.288
9,of,2,1.288


The word **data** appears in all three sentences, so its IDF is 1.0 — the floor, given scikit-learn's
smoothing. It carries no discriminating information, and in s1 and s2 it is duly pushed down the
ranking despite being the topic word. This is TF-IDF behaving as intended: in a corpus entirely about
data science, "data" tells you nothing about *which* sentence you are looking at.

The top terms are otherwise the ones unique or near-unique to a sentence:

- **s1**: `science` and `of`, tied at 0.488, then `most`, `fields`, `important`
- **s2**: `best`, `courses`, `this`, tied at 0.400 — the words that make it a sentence about a course
- **s3**: `data` at 0.641, then `analyze` and `scientists`

Two of those deserve comment, because both cut against the tidy version of the story.

`of` ranks joint-top in s1 purely because it occurs twice. It is a pure function word with no topical
content whatsoever, and IDF only discounts it to 1.288 because s3 happens not to contain it. On a
three-document corpus, IDF has almost no data to work with, and a stopword list would do the job that
IDF is failing to do here.

`data` is the *highest*-weighted term in s3, the one place the "common words get discounted" summary
breaks. s3 is only four tokens long and two of them are `data`, so the term-frequency half of the
product overwhelms the minimum IDF. TF-IDF is a product of two factors, and a large enough TF wins
even at the IDF floor.

Note that `science` and `scientists` are separate terms with separate weights. TF-IDF does no
stemming by default, so morphological variants of the same root compete against each other rather
than reinforcing each other. On a real corpus that is usually worth fixing with a stemmer or
lemmatiser before vectorising.

## Task 3 — Word2Vec on the Simpsons script

`simpsons_script_lines.csv` holds one row per spoken line. Only the `spoken_words` column is used;
`raw_character_text` is the speaker and is not part of the text being modelled.

In [8]:
csv_path = Path("dataset/simpsons_script_lines.csv")

data = pd.read_csv(csv_path, engine="python", on_bad_lines="skip")
print("rows:", len(data))
print("columns:", list(data.columns))
data.head()

rows: 158314
columns: ['raw_character_text', 'spoken_words']


,raw_character_text,spoken_words
0,Miss Hoover,"No, actually, it was a little of both. Sometim..."
1,Lisa Simpson,Where's Mr. Bergstrom?
2,Miss Hoover,I don't know. Although I'd sure like to talk t...
3,Lisa Simpson,That life is worth living.
4,Edna Krabappel-Flanders,The polls will be open from now until the end ...


In [9]:
print("missing spoken_words:", data["spoken_words"].isna().sum())

lines = data["spoken_words"].dropna()
print("usable lines:", len(lines))

missing spoken_words: 26459
usable lines: 131855


About one row in six has no `spoken_words` at all — those are stage directions and scene descriptions
rather than dialogue, and they are dropped.

The cleaning function is the one specified in the lab: lowercase, strip digits, strip a short list of
punctuation. It is worth being clear about what it does *not* remove — `?`, `!`, `:`, `;` and `"`
survive it, so tokens like `hey!` and `hey` end up as different words. I keep the specified function
so the results are comparable to the lab's, then split on whitespace and drop any token that still
holds a non-letter character, which cleans up the residue without changing the approach.

In [10]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,—.'$-]", "", text)
    return text


sample = lines.iloc[0]
print("before:", sample)
print("after: ", clean_text(sample))

before: No, actually, it was a little of both. Sometimes when a disease is in all the magazines and all the news shows, it's only natural that you think you have it.
after:  no actually it was a little of both sometimes when a disease is in all the magazines and all the news shows its only natural that you think you have it


In [11]:
tokens = []
for line in lines:
    words = [w for w in clean_text(line).split() if w.isalpha()]
    if words:
        tokens.append(words)

lengths = np.array([len(t) for t in tokens])
print("sentences:", len(tokens))
print("total tokens:", lengths.sum())
print("vocabulary:", len(set(w for t in tokens for w in t)))
print("median line length:", int(np.median(lengths)), "words")
print("longest line:", lengths.max(), "words")

sentences: 126951
total tokens: 1207397
vocabulary: 40748
median line length: 7 words
longest line: 119 words


In [12]:
import gensim

Skip_gram_model = gensim.models.Word2Vec(
    tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1,
    seed=42,
    workers=1,   # see note below: required for reproducible vectors
)

print(Skip_gram_model)
print("vectors trained for", len(Skip_gram_model.wv), "words")

Word2Vec<vocab=40748, vector_size=100, alpha=0.025>
vectors trained for 40748 words


`seed=42, workers=1` are additions to the constructor the lab gives. They matter more than they look.

With gensim's default of multiple worker threads, training is **not reproducible** even with a fixed
seed — the threads consume training examples in a nondeterministic order, so every run produces
different vectors. I found this the direct way: running the notebook twice gave `homer`'s nearest
neighbour as `abe` the first time and `marge` the second, and moved similarity scores by up to ±0.03.

That is not a large drift and the conclusions below survive it, but any specific number quoted in the
commentary would be wrong on the next run. Pinned to one thread, two separate training runs now agree
to six decimal places. The cost is training speed, which is irrelevant on a corpus this small.

`min_count=1` is what the lab specifies, and it means every single word is kept — including the
thousands that appear exactly once. Those words get a vector, but it is trained from one context
window and is essentially noise. It does not corrupt the frequent words, but it does inflate the
vocabulary and the training time, and it means `most_similar` can occasionally return a word the
model has almost no evidence about. A `min_count` of 5 would be the normal choice on a corpus this
size.

In [13]:
singletons = sum(1 for w in Skip_gram_model.wv.index_to_key
                 if Skip_gram_model.wv.get_vecattr(w, "count") == 1)
print(f"words seen exactly once: {singletons:,} "
      f"({singletons / len(Skip_gram_model.wv):.0%} of the vocabulary)")

print("\nmost frequent words:")
for word in Skip_gram_model.wv.index_to_key[:12]:
    print(f"  {Skip_gram_model.wv.get_vecattr(word, 'count'):>7,}  {word}")

words seen exactly once: 19,366 (48% of the vocabulary)

most frequent words:
   41,359  the
   35,798  you
   35,423  i
   30,934  a
   27,065  to
   19,161  and
   16,147  of
   12,632  my
   12,416  in
   12,144  is
   12,097  it
   11,921  that


## Task 4 — Words similar to the main characters

In [14]:
for name in ["homer", "marge", "bart"]:
    count = Skip_gram_model.wv.get_vecattr(name, "count")
    print(f"{name}  (appears {count:,} times)")
    for word, score in Skip_gram_model.wv.most_similar(name):
        print(f"   {score:.3f}  {word}")
    print()

homer  (appears 3,364 times)
   0.867  marge
   0.861  abe
   0.836  bart
   0.834  grampa
   0.832  barney
   0.827  bartholomew
   0.821  monty
   0.819  eliza
   0.813  ralph
   0.812  apu

marge  (appears 2,250 times)
   0.879  abe
   0.867  homer
   0.846  sweetie
   0.837  sweetheart
   0.837  honey
   0.824  fellas
   0.822  snuggle
   0.822  lisa
   0.820  maude
   0.817  gal

bart  (appears 2,619 times)
   0.873  milhouse
   0.872  lisa
   0.859  grampa
   0.851  abe
   0.839  ralph
   0.836  homer
   0.836  nelson
   0.834  jessica
   0.832  maggie
   0.828  eliza



The neighbours are mostly other character names, which is the result you want: Word2Vec places words
in similar positions when they appear in similar contexts, and character names occupy the same
grammatical slots — being addressed, being talked about, being the subject of a sentence. The model
has learned the *category* "person in this show" without ever being told that such a category exists.

Marge's list is the most informative of the three, because it is not purely names: `sweetie`,
`honey`, `sweetheart` and `homie` sit high in it. Those are vocatives — words used to address
someone — and they fill the same slot in a sentence as a name does. The model has grouped them with
the names because, distributionally, that is what they are.

There is one result that does look like real knowledge: `homer`'s nearest neighbour of all is
`marge`, at 0.867. It is tempting to read that as the model learning they are married. The more
likely explanation is that they are the two characters who most often address each other directly, so
their contexts genuinely do overlap more than any other pair's.

Against that, the model clearly has **not** learned the family structure in general. `bart`'s nearest
neighbour is `milhouse`, his friend, ahead of his own sister; `abe` ranks second for both Homer and
Marge. The grid below puts every family pair in the 0.71–0.87 band with no structure matching the
family tree — and the lowest score of all, 0.714, is Homer and Maggie, who are father and daughter
but rarely exchange dialogue since Maggie does not speak. That is the giveaway that this measures
interchangeability of context, not relationships in the story.

In [15]:
# Does the model place the family closer together than random name pairs?
family = ["homer", "marge", "bart", "lisa", "maggie"]

grid = pd.DataFrame(
    [[Skip_gram_model.wv.similarity(a, b) for b in family] for a in family],
    index=family,
    columns=family,
)
grid.round(3)

,homer,marge,bart,lisa,maggie
homer,1.000,0.867,0.836,0.739,0.714
marge,0.867,1.000,0.807,0.822,0.785
bart,0.836,0.807,1.000,0.872,0.832
lisa,0.739,0.822,0.872,1.000,0.848
maggie,0.714,0.785,0.832,0.848,1.000


## Task 5 — `doesnt_match`

`doesnt_match` averages the vectors in the list and returns the word furthest from that average. It
always returns something, even when the list has no meaningful odd one out — so the output needs to
be read alongside the actual similarity scores rather than taken at face value.

In [16]:
questions = [
    ["jimbo", "milhouse", "kearney"],
    ["nelson", "bart", "milhouse"],
    ["homer", "patty", "selma"],
]

for words in questions:
    odd = Skip_gram_model.wv.doesnt_match(words)
    print(f"{words}  ->  {odd}")

['jimbo', 'milhouse', 'kearney']  ->  milhouse
['nelson', 'bart', 'milhouse']  ->  nelson
['homer', 'patty', 'selma']  ->  homer


In [17]:
# The pairwise scores behind each answer.
for words in questions:
    print(words)
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            print(f"   {Skip_gram_model.wv.similarity(words[i], words[j]):.3f}  "
                  f"{words[i]} / {words[j]}")
    print(f"   -> odd one out: {Skip_gram_model.wv.doesnt_match(words)}")
    print()

['jimbo', 'milhouse', 'kearney']
   0.875  jimbo / milhouse
   0.880  jimbo / kearney
   0.815  milhouse / kearney
   -> odd one out: milhouse

['nelson', 'bart', 'milhouse']
   0.836  nelson / bart
   0.861  nelson / milhouse
   0.873  bart / milhouse
   -> odd one out: nelson

['homer', 'patty', 'selma']
   0.687  homer / patty
   0.704  homer / selma
   0.926  patty / selma
   -> odd one out: homer



All three answers match the characters as they are written in the show, which is more than I expected
from a model given nothing but co-occurrence counts.

- **jimbo / milhouse / kearney → milhouse.** Jimbo and Kearney are bullies and score 0.880 with each
  other; Milhouse, who is not, drops to 0.815 against Kearney.
- **nelson / bart / milhouse → nelson.** Bart and Milhouse are friends and score 0.873; Nelson, the
  bully, is the one attached to neither.
- **homer / patty / selma → homer.** The clearest case by far. Patty and Selma are twins who appear
  together constantly and score **0.926**, while Homer sits at 0.687 and 0.704 against them.

The margins are what separate a trustworthy answer from a lucky one. The third question is decided by
a gap of about **0.22**, which is not close. The first two turn on gaps of **0.037 and 0.065** — real,
but thin enough that I would not bet on them surviving a change of random seed. Before pinning the
seed I saw these same two lists move by ±0.03 between runs, which is the same order as the margin
itself.

That is the standing caveat on `doesnt_match`: it returns the word furthest from the mean no matter
what, and has no way to report that the margin was 0.006 or that the honest answer is "these are all
the same kind of thing". The pairwise scores have to be read alongside it.

## Summary

- Cosine similarity on TF-IDF vectors correctly identified documents 1 and 4 as the closest pair, and
  correctly isolated document 3. It also demonstrated the bag-of-words blind spot: a sentence and its
  question form are identical to the model.
- On the data-science sentences, `data` was pushed to the IDF floor of 1.0 for appearing everywhere —
  but still came out top in the shortest sentence, because a high enough term frequency beats the
  minimum IDF. The rule is a tendency, not a guarantee.
- A skip-gram model trained on 127k lines of Simpsons dialogue learned to group character names
  together from context alone, and to group terms of address (`honey`, `sweetie`) with them.
- `doesnt_match` answered all three questions correctly, but with very different margins: 0.22 for
  Homer / Patty / Selma versus 0.037 and 0.065 for the two lists of schoolboys. The pairwise
  similarities, not the returned word, are what tell you which answers to trust.
- Word2Vec training is nondeterministic under gensim's default thread count. Fixing `seed` alone is
  not enough; `workers=1` is also required, and without it no number quoted above would be stable.